### **Description:**

* Pick a domain (except movies and restaurants) in which recommendations makes sense.     


* Create a Colab  notebook with four functions:


    1. generate_entities(): A function that can generate recommendable entities. These entities should have at least 10 distinct attributes. The number of entities should be a parameter


    2. generate_users(): A function that can generate users, with each user belonging to one of 5 customer segments with distinct personalities. Each personality should be based on a combination of at least 5 different entity attributes that determine whether the user will like or dislike a specific entity if it was recommended to them.  The number of users should be a parameter. The internal documentation of this function should include a short paragraph that describes ther personality of each segment. The creativity of the personalities is important for this assignment. 


    3. generate_ratings(): A function that generates binary like/dislike  ratings given a set of users and a set of entities. The number of ratings should be a parameter. Another parameter should be a noise percentage, that determines whether a user will like/dislike an entity based on his personality or based on chance. 


    4. learn_segments(): A function that reads the ratings generated by the third function and uses them to identify and print the underlying customer segments.
 
_Notes_: Submit the link to your Colab notebook. Make sure it's publicly accessible. Make sure that 4 functions are named exactly as defined above.


## **Name:** Michail Theofanopoulos &nbsp;·&nbsp; p3352401

### Approach - What this notebook does

This notebook simulates a recommender system for fictional NBA players. We generate players with realistic stats, create fans belonging to 5 hidden personality segments, simulate their ratings and then run 5 different algorithms to recover those segments from the ratings alone without ever showing the algorithms which segment each fan belongs to.

Performance is measured with ARI (Adjusted Rand Index): 0 = random, 1 = perfect.

In [181]:
from dataclasses import dataclass, field
from typing import List, Optional
import numpy as np
import pandas as pd
import random
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import StandardScaler, LabelEncoder
from gensim.models import Word2Vec
from minisom import MiniSom

### Setup

The 15 player attributes are ordered by how useful they are for distinquishing fan segments. The `num_attributes` parameter exploits this ordering: using only the first 5 gives a harder problem, using all 15 gives maximum signal.

`Player` and `Fan` are data containers. The `segment` field on `Fan` is the ground truth kept for evaluation but hidden from every algorithm.

In [182]:
MASTER_ATTRIBUTES = [
    "blocks_per_game",
    "assists_per_game",
    "points_per_game",
    "field_goal_pct",
    "all_star_appearances",
    "salary_tier",
    "three_point_pct",
    "position",
    "steals_per_game",
    "free_throw_pct",
    "team",
    "rebounds_per_game",
    "playstyle",
    "age",
    "nationality",
]

POSITIONS     = ["PG", "SG", "SF", "PF", "C"]
SALARY_TIERS  = ["max", "mid", "minimum"]
PLAYSTYLES    = ["scorer", "playmaker", "defender", "versatile", "role_player"]
NATIONALITIES = ["domestic", "international"]
TEAMS = [
    "Philadelphia 76ers", "Milwaukee Bucks", "Chicago Bulls", "Charlotte Hornets",
    "Miami Heat", "Los Angeles Lakers", "Los Angeles Clippers", "Toronto Raptors",
    "Houston Rockets", "Dallas Mavericks", "Golden State Warriors", "Utah Jazz"
]

In [183]:
@dataclass
class Player:
    player_id: int
    name: str
    blocks_per_game: float      # Gaussian(0.8, 0.7), clip 0–5
    assists_per_game: float     # Gaussian(4, 3), clip 0–15
    points_per_game: float      # Gaussian(12, 6), clip 0–45
    field_goal_pct: float       # Gaussian(0.46, 0.08), clip 0–0.7
    all_star_appearances: int   # 0–15, correlated with PPG
    salary_tier: str            # "max"(10%), "mid"(40%), "minimum"(50%)
    three_point_pct: float      # Gaussian(0.35, 0.08), clip 0–0.6
    position: str               # "PG","SG","SF","PF","C" — uniform
    steals_per_game: float      # Gaussian(1.0, 0.8), clip 0–4
    free_throw_pct: float       # Gaussian(0.75, 0.12), clip 0–1
    team: str                   # random from TEAMS
    rebounds_per_game: float    # Gaussian(5, 3), clip 0–20
    playstyle: str              # "scorer","playmaker","defender","versatile","role_player"
    age: int                    # Gaussian(26, 4), clip 18–40
    nationality: str            # "domestic"(70%) / "international"(30%)

@dataclass
class Fan:
    fan_id: int
    segment: int                # 1–5 (ground truth, NOT passed to learn_segments)
    age: int

In [184]:
def generate_entities(n: int=100, num_attributes: int=15) -> pd.DataFrame:
    """
    Generates a DataFrame of fictional NBA players.

    Attributes are ordered by discriminability (most informative first).
    num_attributes controls difficulty: fewer attributes = harder segment recovery.
    Attribute 15 (nationality) is a red herring — no segment uses it.

    Correlations enforced:
    - all_star_appearances ~ Normal(PPG/5, 1), clipped to [0, 15]
    - salary_tier probabilities shift with PPG (high scorers more likely "max")

    Args:
        n (int): Number of players to generate.
        num_attributes (int): Active attributes to include, 1–15.
    Returns:
        pd.DataFrame: Shape (n, 2+num_attributes) with player_id, name, and active attributes.
    """

    players = []

    for i in range(n):
        ppg = float(np.clip(np.random.normal(12, 6), 0, 45))

        # all_star_appearances correlated with PPG
        stars = int(np.clip(round(np.random.normal(ppg / 5, 1)), 0, 15))

        # salary_tier correlated with PPG
        if ppg >= 22:
            salary = np.random.choice(SALARY_TIERS, p=[0.70, 0.25, 0.05])
        elif ppg >= 12:
            salary = np.random.choice(SALARY_TIERS, p=[0.15, 0.60, 0.25])
        else:
            salary = np.random.choice(SALARY_TIERS, p=[0.02, 0.28, 0.70])

        p = Player(
            player_id           = i,
            name                = f"Player_{i}",
            blocks_per_game     = round(float(np.clip(np.random.normal(0.8, 0.7), 0, 5)), 2),
            assists_per_game    = round(float(np.clip(np.random.normal(4, 3), 0, 15)), 2),
            points_per_game     = round(ppg, 2),
            field_goal_pct      = round(float(np.clip(np.random.normal(0.46, 0.08), 0, 0.7)), 2),
            all_star_appearances= stars,
            salary_tier         = salary,
            three_point_pct     = round(float(np.clip(np.random.normal(0.35, 0.08), 0, 0.6)), 2),
            position            = random.choice(POSITIONS),
            steals_per_game     = round(float(np.clip(np.random.normal(1.0, 0.8), 0, 4))),
            free_throw_pct      = round(float(np.clip(np.random.normal(0.75, 0.12), 0, 1.0)), 2),
            team                = random.choice(TEAMS),
            rebounds_per_game   = round(float(np.clip(np.random.normal(5, 3), 0, 20)), 2),
            playstyle           = random.choice(PLAYSTYLES),
            age                 = int(np.clip(round(np.random.normal(26, 4)), 18, 40)),
            nationality         = np.random.choice(NATIONALITIES, p=[0.70, 0.30]),
        )
        players.append(p)

    df = pd.DataFrame([vars(p) for p in players])

    # Keep only player_id, name, and the first num_attributes active columns
    active_cols = MASTER_ATTRIBUTES[:num_attributes]
    return df[["player_id", "name"] + active_cols]

### Data Generation

The three functions below build the simulated world. `generate_entities` creates the player table, `generate_users` creates the fan base and `generate_ratings` simulates which players each fan would like based on their hidden personality type.

Each segment's preference rule has a **core condition** (required to like) and up to three **boost attributes** (each raises the like probability by +10%). This ensures every segment uses at least 5 distinct attributes.

| Segment | Core rule | Boost attributes (+10% each) |
|---------|-----------|-------------------------------|
| Stat Nerds | field_goal_pct > 0.50 AND salary_tier ∈ {mid, minimum} | free_throw_pct > 0.78, rebounds_per_game > 4.0, all_star_appearances < 3 |
| Highlight Reel | points_per_game > 20 OR all_star_appearances > 3 | three_point_pct > 0.38, salary_tier = max, age < 28 |
| Defense Purists | blocks_per_game > 1.0 AND steals_per_game > 1.0 | position ∈ {C, PF}, rebounds_per_game > 6.0, age > 24 |
| Veteran Appreciators | age > 30 AND free_throw_pct > 0.80 | all_star_appearances > 0, points_per_game > 8, salary_tier ∈ {mid, minimum} |
| Playmaker Devotees | assists_per_game > 6 AND position = PG | field_goal_pct > 0.44, steals_per_game > 0.8, free_throw_pct > 0.75 |

In [185]:
def generate_users(segment_sizes: List[int] = [2000, 1500, 1000, 800, 500]) -> pd.DataFrame:
    """
    Generates fans across 5 segments with distinct personalities.

    Segment 1 — Stat Nerds (age ~ N(30,5)):
        Analytically minded fans who value shooting efficiency and contract value above
        highlights. They prize an efficient player on a mid-level or minimum deal, using
        field goal percentage and salary tier as primary filters.

    Segment 2 — Highlight Reel (age ~ N(20,5)):
        Entertainment-first fans drawn to stars and big moments. A superstar scorer or
        decorated all-star earns their attention immediately; three-point shooting, max
        contracts, and youth add extra appeal.

    Segment 3 — Defense Purists (age ~ N(35,8)):
        Traditionalist fans who believe defense wins championships. They require both
        shot-blocking and stealing — one defensive dimension alone is not enough. Centers
        and power forwards who anchor the paint are the archetype.

    Segment 4 — Veteran Appreciators (age ~ N(40,8)):
        Mature fans who respect longevity and professionalism over athleticism. Their ideal
        player is an experienced veteran over 30 who shoots free throws reliably — a proxy
        for composure — and still contributes without commanding a max contract.

    Segment 5 — Playmaker Devotees (age ~ N(28,6)):
        Basketball purists who view the point guard as the most important position. They
        prize elite passers who run an offense, specifically PGs averaging over 6 assists,
        complemented by shooting efficiency and active hands.

    Args:
        segment_sizes (List[int]): Number of fans per segment (default: [2000,1500,1000,800,500]).
    Returns:
        pd.DataFrame: Shuffled fan DataFrame with columns fan_id, segment, age.
    """

    age_dist = {1: (30, 5), 2: (20, 5), 3: (35, 8), 4: (40, 8), 5: (28, 6)}
    fans   = []
    fan_id = 0

    for segment, n in enumerate(segment_sizes, start=1):
        mean_age, std_age = age_dist[segment]
        for _ in range(n):
            fans.append(Fan(
                fan_id  = fan_id,
                segment = segment,
                age     = int(np.clip(round(np.random.normal(mean_age, std_age)), 15, 70)),
            ))
            fan_id += 1

    df = pd.DataFrame([vars(f) for f in fans])
    print(f"{len(df)} fans  ({' / '.join(str(n) for n in segment_sizes)})")
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
def generate_ratings(players_df, users_df, n_ratings_per_user=50, noise_pct=0.10):
    """
    Generates binary like/dislike ratings for each (fan, player) pair.

    Each segment uses a core condition (required to like) plus up to 3 boost attributes
    (+10% each). Base like probability when core is met: 0.65 (max 0.95 with all boosts).
    Per-segment noise rates: Stat Nerds 10%, Highlight Reel 20%, Defense Purists 15%,
    Veteran Appreciators 10%, Playmaker Devotees 20%.
    If a required attribute is absent, the fan rates randomly.

    Args:
        players_df (pd.DataFrame): Output of generate_entities().
        users_df (pd.DataFrame):   Output of generate_users().
        n_ratings_per_user (int):  Players sampled per fan (default: 50).
        noise_pct (float):         Fallback noise if segment not found (default: 0.10).
    Returns:
        pd.DataFrame: Shuffled ratings with fan features, player features, rating (-1/1), reason.
    """

    players = players_df.to_dict('records')
    rows    = []
    noise_per_segment = {1: 0.10, 2: 0.20, 3: 0.15, 4: 0.10, 5: 0.20}

    for _, fan in users_df.iterrows():
        seg     = fan['segment']
        flip_p  = noise_per_segment.get(seg, noise_pct)
        sampled = random.sample(players, min(n_ratings_per_user, len(players)))

        for p in sampled:
            rating = -1
            reason = ""

            # Segment 1: Stat Nerds
            if seg == 1:
                if 'field_goal_pct' in p and 'salary_tier' in p:
                    core = p['field_goal_pct'] > 0.50 and p['salary_tier'] in ('mid', 'minimum')
                    if core:
                        boost = sum([
                            'free_throw_pct'      in p and p['free_throw_pct']      > 0.78,
                            'rebounds_per_game'   in p and p['rebounds_per_game']   > 4.0,
                            'all_star_appearances' in p and p['all_star_appearances'] < 3,
                        ])
                        if random.random() < 0.65 + 0.10 * boost:
                            rating = 1
                            reason = (f"Efficient value pick (FG%={p['field_goal_pct']:.2f}, "
                                      f"{p['salary_tier']}, boosts={boost})")
                        else:
                            reason = "Core met but secondary profile insufficient"
                    else:
                        reason = f"Not efficient or too expensive (FG%={p['field_goal_pct']:.2f}, {p['salary_tier']})"
                else:
                    reason = "Missing key attributes"

            # Segment 2: Highlight Reel
            elif seg == 2:
                if 'points_per_game' in p and 'all_star_appearances' in p:
                    core = p['points_per_game'] > 20 or p['all_star_appearances'] > 3
                    if core:
                        boost = sum([
                            'three_point_pct' in p and p['three_point_pct'] > 0.38,
                            'salary_tier'     in p and p['salary_tier'] == 'max',
                            'age'             in p and p['age'] < 28,
                        ])
                        if random.random() < 0.45 + 0.10 * boost:
                            rating = 1
                            reason = (f"Star player (PPG={p['points_per_game']:.2f}, "
                                      f"stars={p['all_star_appearances']}, boosts={boost})")
                        else:
                            reason = "Star but not compelling enough"
                    else:
                        reason = f"Not a star (PPG={p['points_per_game']:.2f}, stars={p['all_star_appearances']})"
                else:
                    reason = "Missing key attributes"

            # Segment 3: Defense Purists
            elif seg == 3:
                if 'blocks_per_game' in p and 'steals_per_game' in p:
                    core = p['blocks_per_game'] > 1.0 and p['steals_per_game'] > 1.0
                    if core:
                        boost = sum([
                            'position'          in p and p['position'] in ('C', 'PF'),
                            'rebounds_per_game' in p and p['rebounds_per_game'] > 6.0,
                            'age'               in p and p['age'] > 24,
                        ])
                        if random.random() < 0.65 + 0.10 * boost:
                            rating = 1
                            reason = (f"Solid defender (BLK={p['blocks_per_game']:.2f}, "
                                      f"STL={p['steals_per_game']:.2f}, boosts={boost})")
                        else:
                            reason = "Core met but not the complete defensive profile"
                    else:
                        reason = f"Insufficient defense (BLK={p['blocks_per_game']:.2f}, STL={p['steals_per_game']:.2f})"
                else:
                    reason = "Missing key attributes"

            # Segment 4: Veteran Appreciators
            elif seg == 4:
                if 'age' in p and 'free_throw_pct' in p:
                    core = p['age'] > 30 and p['free_throw_pct'] > 0.80
                    if core:
                        boost = sum([
                            'all_star_appearances' in p and p['all_star_appearances'] > 0,
                            'points_per_game'      in p and p['points_per_game']      > 8,
                            'salary_tier'          in p and p['salary_tier'] in ('mid', 'minimum'),
                        ])
                        if random.random() < 0.65 + 0.10 * boost:
                            rating = 1
                            reason = (f"Reliable veteran (age={p['age']}, "
                                      f"FT%={p['free_throw_pct']:.2f}, boosts={boost})")
                        else:
                            reason = "Veteran but secondary profile lacking"
                    else:
                        reason = f"Too young or unreliable (age={p['age']}, FT%={p['free_throw_pct']:.2f})"
                else:
                    reason = "Missing key attributes"

            # Segment 5: Playmaker Devotees
            elif seg == 5:
                if 'assists_per_game' in p and 'position' in p:
                    core = p['assists_per_game'] > 6 and p['position'] == 'PG'
                    if core:
                        boost = sum([
                            'field_goal_pct'   in p and p['field_goal_pct']   > 0.44,
                            'steals_per_game'  in p and p['steals_per_game']  > 0.8,
                            'free_throw_pct'   in p and p['free_throw_pct']   > 0.75,
                        ])
                        if random.random() < 0.65 + 0.10 * boost:
                            rating = 1
                            reason = (f"Elite floor general (AST={p['assists_per_game']:.2f}, "
                                      f"PG, boosts={boost})")
                        else:
                            reason = "PG playmaker but secondary profile lacking"
                    else:
                        reason = f"Not a true playmaker (AST={p['assists_per_game']:.2f}, pos={p.get('position','?')})"
                else:
                    reason = "Missing key attributes"

            # Apply noise
            if random.random() < flip_p:
                rating *= -1
                reason += " [flipped]"

            row = {'fan_id': fan['fan_id'], 'fan_age': fan['age'], 'fan_segment': fan['segment']}
            row.update({f"player_{k}": v for k, v in p.items()})
            row['rating'] = rating
            row['reason'] = reason
            rows.append(row)

    result_df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)

    pos_rates = "  ".join(
        f"seg{s}: {(result_df[result_df['fan_segment']==s]['rating']==1).mean():.1%}"
        for s in range(1, 6)
    )
    print(f"{len(result_df)} ratings  |  {pos_rates}")

    return result_df

### Segment Recovery

Each algorithm receives the ratings table with `fan_segment` removed and must discover the 5 groups on its own.

After grouping, every method runs the same exact rule extraction step:

- A shallow decision tree is fit on the player features vs. rating for each discovered group and only the branches leading to a positive (liked) outcome are printed. This translates every method's output into the same interpretable format.

In [187]:
def extract_positive_rules(dt, feature_names):
    """Return only the conditions that lead to a liked (class=1) leaf."""
    tree  = dt.tree_
    rules = []

    def traverse(node, conditions):
        is_leaf = tree.children_left[node] == tree.children_right[node]
        if is_leaf:
            if tree.value[node][0].argmax() == 1:
                rules.append(conditions[:])
        else:
            feat   = feature_names[tree.feature[node]]
            thresh = tree.threshold[node]
            traverse(tree.children_left[node],  conditions + [f"{feat} <= {thresh:.2f}"])
            traverse(tree.children_right[node], conditions + [f"{feat} > {thresh:.2f}"])

    traverse(0, [])
    return rules


#### Method 1 - k-Means

Each fan's liked players are averaged into a single preference vector. k-Means partitions fans into 5 clusters by minimizing within-cluster variance on those vectors. It is the natural baseline: fans who liked similar players end up in the same group.

In [ ]:
def learn_segments_kmeans(ratings_df: pd.DataFrame) -> pd.Series:
    """
    Method 1: k-Means on per-fan aggregated preference vectors.
    Each fan's liked players are averaged into a feature vector; k-Means splits
    fans into 5 clusters. A per-cluster Decision Tree then extracts the rules.
    """

    num_cols = [c for c in ratings_df.columns
                if c.startswith('player_')
                and c not in ('player_player_id', 'player_name')
                and pd.api.types.is_numeric_dtype(ratings_df[c])]

    cat_cols = [c for c in ratings_df.columns
                if c.startswith('player_')
                and c not in ('player_player_id', 'player_name')
                and not pd.api.types.is_numeric_dtype(ratings_df[c])]

    # build per-fan preference vector ──────────────────────────
    liked = ratings_df[ratings_df['rating'] == 1].copy()

    num_agg = liked.groupby('fan_id')[num_cols].mean()

    like_rate = (
        ratings_df.groupby('fan_id')['rating']
        .apply(lambda x: (x == 1).mean())
        .rename('like_rate')
    )

    if cat_cols:
        cat_agg = liked.groupby('fan_id')[cat_cols].agg(
            lambda x: x.mode().iloc[0] if len(x) > 0 else np.nan
        )
        for col in cat_cols:
            cat_agg[col] = LabelEncoder().fit_transform(cat_agg[col].astype(str))
        user_features = num_agg.join(cat_agg).join(like_rate).dropna()
    else:
        user_features = num_agg.join(like_rate).dropna()

    X = StandardScaler().fit_transform(user_features)
    km = KMeans(n_clusters=5, random_state=42, n_init=10)
    labels = km.fit_predict(X)

    # extract rules per cluster ────────────────────────────────
    player_df = ratings_df[['fan_id', 'rating'] + num_cols + cat_cols].copy()
    for col in cat_cols:
        player_df[col] = LabelEncoder().fit_transform(player_df[col].astype(str))

    feature_names = [c.replace('player_', '') for c in num_cols + cat_cols]
    y_all = (player_df['rating'] == 1).astype(int)

    print("[k-Means]")
    for i in range(5):
        fan_ids_in_cluster = user_features.index[labels == i]
        mask = player_df['fan_id'].isin(fan_ids_in_cluster)

        X_cluster = player_df.loc[mask, num_cols + cat_cols]
        y_cluster = y_all[mask]

        dt = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=42)
        dt.fit(X_cluster, y_cluster)

        print(f"  group {i+1}  ({len(fan_ids_in_cluster)} fans, {y_cluster.mean():.1%})")
        rules = extract_positive_rules(dt, feature_names)
        if rules:
            for rule in rules:
                print("    " + " AND ".join(rule))
        else:
            print("    no clear preference found")

    return pd.Series(labels, index=user_features.index, name='kmeans_label')

#### Method 2 - NMF

NMF decomposes the full fan×player rating matrix into 5 latent components. Each componnent represents a latent taste profile: each fun is assigned to their dominant component. Unlike k-Means, NMF works directly on the interaction matrix rather than pre-averaged feature vectors.

In [ ]:
def learn_segments_nmf(ratings_df: pd.DataFrame) -> pd.Series:
    """
    Method 2: NMF on the fan×player rating matrix.
    Decomposes R ≈ W×H into 5 components; each fan is assigned to their
    highest-weight component. A per-component Decision Tree extracts rules.
    """

    num_cols = [c for c in ratings_df.columns
                if c.startswith('player_')
                and c not in ('player_player_id', 'player_name')
                and pd.api.types.is_numeric_dtype(ratings_df[c])]

    cat_cols = [c for c in ratings_df.columns
                if c.startswith('player_')
                and c not in ('player_player_id', 'player_name')
                and not pd.api.types.is_numeric_dtype(ratings_df[c])]

    # NMF grouping
    df = ratings_df.copy()
    df['liked'] = (df['rating'] == 1).astype(float)
    R = df.pivot_table(index='fan_id', columns='player_player_id',
                       values='liked', fill_value=0)

    W = NMF(n_components=5, random_state=42, max_iter=500).fit_transform(R)
    labels  = W.argmax(axis=1)
    fan_ids = R.index

    # rule extraction per component
    player_df = ratings_df[['fan_id', 'rating'] + num_cols + cat_cols].copy()
    for col in cat_cols:
        player_df[col] = LabelEncoder().fit_transform(player_df[col].astype(str))

    feature_names = [c.replace('player_', '') for c in num_cols + cat_cols]
    y_all = (player_df['rating'] == 1).astype(int)

    print("[NMF]")
    for i in range(5):
        fan_ids_in_component = fan_ids[labels == i]
        mask = player_df['fan_id'].isin(fan_ids_in_component)

        X_cluster = player_df.loc[mask, num_cols + cat_cols]
        y_cluster = y_all[mask]

        dt = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=42)
        dt.fit(X_cluster, y_cluster)

        print(f"  component {i+1}  ({len(fan_ids_in_component)} fans, {y_cluster.mean():.1%})")
        rules = extract_positive_rules(dt, feature_names)
        if rules:
            for rule in rules:
                print("    " + " AND ".join(rule))
        else:
            print("    no clear preference found")

    return pd.Series(labels, index=fan_ids, name='nmf_label')

#### Method 3 - Item2Vec

Each fan's liked-player sequence is treated as a document, each player ID as a token. Word2Vec learns embeddings so players frequently co-liked by the same fans end up close in vector space. Each fan is represented as the mean of their liked players' embeddings and then clustered.

This is purely collaborative since player attributes are never used in the embedding step. The trade-off: it needs dense interaction data to work. With sparse ratings it collapses.

In [ ]:
def learn_segments_item2vec(ratings_df: pd.DataFrame) -> pd.Series:
    """
    Method 3: Item2Vec (Word2Vec on player co-likes) + k-Means.
    Each fan's liked players form a "sentence"; Word2Vec learns embeddings so
    co-liked players end up close in vector space. Fans are represented as their
    mean player embedding, then clustered. Purely collaborative — no player
    attributes used in the embedding step.
    """

    num_cols = [c for c in ratings_df.columns
                if c.startswith('player_')
                and c not in ('player_player_id', 'player_name')
                and pd.api.types.is_numeric_dtype(ratings_df[c])]

    cat_cols = [c for c in ratings_df.columns
                if c.startswith('player_')
                and c not in ('player_player_id', 'player_name')
                and not pd.api.types.is_numeric_dtype(ratings_df[c])]

    # Item2Vec grouping
    liked_df = ratings_df[ratings_df['rating'] == 1]

    sentences = []
    fan_ids   = []
    for fan_id, group in liked_df.groupby('fan_id'):
        tokens = group['player_player_id'].astype(str).tolist()
        if tokens:
            sentences.append(tokens)
            fan_ids.append(fan_id)

    model = Word2Vec(sentences, vector_size=32, window=10,
                     min_count=1, workers=1, seed=42, epochs=20)

    fan_vecs = []
    valid_fan_ids = []
    for fan_id, tokens in zip(fan_ids, sentences):
        vecs = [model.wv[t] for t in tokens if t in model.wv]
        if vecs:
            fan_vecs.append(np.mean(vecs, axis=0))
            valid_fan_ids.append(fan_id)

    X      = StandardScaler().fit_transform(np.array(fan_vecs))
    km     = KMeans(n_clusters=5, random_state=42, n_init=10)
    labels = km.fit_predict(X)

    # rule extraction per cluster
    player_df = ratings_df[['fan_id', 'rating'] + num_cols + cat_cols].copy()
    for col in cat_cols:
        player_df[col] = LabelEncoder().fit_transform(player_df[col].astype(str))

    feature_names = [c.replace('player_', '') for c in num_cols + cat_cols]
    y_all = (player_df['rating'] == 1).astype(int)

    print("[Item2Vec]")
    for i in range(5):
        fan_ids_in_cluster = [fid for fid, lbl in zip(valid_fan_ids, labels) if lbl == i]
        mask = player_df['fan_id'].isin(fan_ids_in_cluster)

        X_cluster = player_df.loc[mask, num_cols + cat_cols]
        y_cluster = y_all[mask]

        dt = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=42)
        dt.fit(X_cluster, y_cluster)

        print(f"  cluster {i+1}  ({len(fan_ids_in_cluster)} fans, {y_cluster.mean():.1%})")
        rules = extract_positive_rules(dt, feature_names)
        if rules:
            for rule in rules:
                print("    " + " AND ".join(rule))
        else:
            print("    no clear preference found")

    return pd.Series(labels, index=valid_fan_ids, name='item2vec_label')

#### Method 4 - Self-Organizing Map

A SOM projects fans into a 10×10 grid where similar fans occupy neighboring cells. The topology constraint means fans who sit between two segment types land at the bounday on the grid, making overlap geometrically visible. The grid cells are then grouped into 5 regions with k-Means.

In [ ]:
def learn_segments_som(ratings_df: pd.DataFrame) -> pd.Series:
    """
    Method 4: Self-Organizing Map + k-Means on the grid.
    Per-fan preference vectors are projected onto a 10×10 topology-preserving
    grid. Fans with similar tastes land on neighboring cells. The winning neuron
    coordinates are then clustered into 5 regions with k-Means.
    """

    num_cols = [c for c in ratings_df.columns
                if c.startswith('player_')
                and c not in ('player_player_id', 'player_name')
                and pd.api.types.is_numeric_dtype(ratings_df[c])]

    cat_cols = [c for c in ratings_df.columns
                if c.startswith('player_')
                and c not in ('player_player_id', 'player_name')
                and not pd.api.types.is_numeric_dtype(ratings_df[c])]

    # build per-fan preference vectors
    liked = ratings_df[ratings_df['rating'] == 1].copy()
    num_agg = liked.groupby('fan_id')[num_cols].mean()

    like_rate = (
        ratings_df.groupby('fan_id')['rating']
        .apply(lambda x: (x == 1).mean())
        .rename('like_rate')
    )

    if cat_cols:
        cat_agg = liked.groupby('fan_id')[cat_cols].agg(
            lambda x: x.mode().iloc[0] if len(x) > 0 else np.nan
        )
        for col in cat_cols:
            cat_agg[col] = LabelEncoder().fit_transform(cat_agg[col].astype(str))
        user_features = num_agg.join(cat_agg).join(like_rate).dropna()
    else:
        user_features = num_agg.join(like_rate).dropna()

    X = StandardScaler().fit_transform(user_features)
    n_features = X.shape[1]

    # Train SOM
    som = MiniSom(10, 10, n_features, sigma=1.5, learning_rate=0.5,
                  neighborhood_function='gaussian', random_seed=42)
    som.random_weights_init(X)
    som.train(X, num_iteration=5000, verbose=False)

    winner_coords = np.array([som.winner(x) for x in X])

    km     = KMeans(n_clusters=5, random_state=42, n_init=10)
    labels = km.fit_predict(winner_coords)

    # rule extraction per region
    player_df = ratings_df[['fan_id', 'rating'] + num_cols + cat_cols].copy()
    for col in cat_cols:
        player_df[col] = LabelEncoder().fit_transform(player_df[col].astype(str))

    feature_names = [c.replace('player_', '') for c in num_cols + cat_cols]
    y_all = (player_df['rating'] == 1).astype(int)

    print("[SOM]")
    for i in range(5):
        fan_ids_in_region = user_features.index[labels == i]
        mask = player_df['fan_id'].isin(fan_ids_in_region)

        X_cluster = player_df.loc[mask, num_cols + cat_cols]
        y_cluster = y_all[mask]

        dt = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=42)
        dt.fit(X_cluster, y_cluster)

        print(f"  region {i+1}  ({len(fan_ids_in_region)} fans, {y_cluster.mean():.1%})")
        rules = extract_positive_rules(dt, feature_names)
        if rules:
            for rule in rules:
                print("    " + " AND ".join(rule))
        else:
            print("    no clear preference found")

    return pd.Series(labels, index=user_features.index, name='som_label')

#### Method 5 - Decision Tree

k-Means labels are used as pseudo labels to train a multi-class decision tree on per-fan preference vectors. The tree outputs explicit IF-THEN rules and feature importances which is the most interpretable result of any method. ARI is structurally identical to k-Means since the labels are inherited from it.

In [ ]:
def learn_segments_dt(ratings_df: pd.DataFrame) -> pd.Series:
    """
    Method 5: Decision Tree on per-fan preference vectors.
    k-Means pseudo-labels are used as supervision for a multi-class Decision Tree.
    Outputs explicit IF-THEN rules and feature importances — the most interpretable
    method. ARI mirrors k-Means since labels are inherited from it.
    """

    num_cols = [c for c in ratings_df.columns
                if c.startswith('player_')
                and c not in ('player_player_id', 'player_name')
                and pd.api.types.is_numeric_dtype(ratings_df[c])]

    cat_cols = [c for c in ratings_df.columns
                if c.startswith('player_')
                and c not in ('player_player_id', 'player_name')
                and not pd.api.types.is_numeric_dtype(ratings_df[c])]

    # per-fan preference vectors
    liked = ratings_df[ratings_df['rating'] == 1].copy()

    num_agg = liked.groupby('fan_id')[num_cols].mean()

    like_rate = (
        ratings_df.groupby('fan_id')['rating']
        .apply(lambda x: (x == 1).mean())
        .rename('like_rate')
    )

    if cat_cols:
        cat_agg = liked.groupby('fan_id')[cat_cols].agg(
            lambda x: x.mode().iloc[0] if len(x) > 0 else np.nan
        )
        for col in cat_cols:
            cat_agg[col] = LabelEncoder().fit_transform(cat_agg[col].astype(str))
        user_features = num_agg.join(cat_agg).join(like_rate).dropna()
    else:
        user_features = num_agg.join(like_rate).dropna()

    feature_names = [c.replace('player_', '') for c in user_features.columns]

    X  = StandardScaler().fit_transform(user_features)
    km = KMeans(n_clusters=5, random_state=42, n_init=10)
    pseudo_labels = km.fit_predict(X)

    # multi-class DT on fan-level features
    dt = DecisionTreeClassifier(max_depth=4, min_samples_leaf=30, random_state=42)
    dt.fit(user_features, pseudo_labels)

    tree = dt.tree_

    def extract_class_rules(node, conditions):
        is_leaf = tree.children_left[node] == tree.children_right[node]
        if is_leaf:
            cls = int(tree.value[node][0].argmax())
            class_rules[cls].append(conditions[:])
        else:
            feat   = feature_names[tree.feature[node]]
            thresh = tree.threshold[node]
            extract_class_rules(tree.children_left[node],  conditions + [f"{feat} <= {thresh:.2f}"])
            extract_class_rules(tree.children_right[node], conditions + [f"{feat} > {thresh:.2f}"])

    class_rules = {i: [] for i in range(5)}
    extract_class_rules(0, [])

    print("[Decision Tree]")
    for i in range(5):
        size = (pseudo_labels == i).sum()
        print(f"  segment {i+1}  ({size} fans)")
        if class_rules[i]:
            for rule in class_rules[i]:
                print("    " + " AND ".join(rule))
        else:
            print("    no distinct rule found")

    importances = pd.Series(dt.feature_importances_, index=feature_names)
    top = importances.nlargest(6)
    print(f"\n  feature importances: " + "  ".join(f"{f} {v:.3f}" for f, v in top.items()))

    return pd.Series(pseudo_labels, index=user_features.index, name='dt_label')

In [193]:
def learn_segments(ratings_df: pd.DataFrame,
                   method: str    = 'kmeans',
                   min_liked: int = 5) -> pd.Series:
    """
    Identifies the underlying customer segments from a ratings table.

    Args:
    - ratings_df:  Output of generate_ratings(), ground truth columns removed.
    - method:      Segmentation method — one of: 'kmeans', 'nmf', 'item2vec', 'som', 'dt'
    - min_liked:   Minimum liked ratings a fan must have to be included (default: 5).

    Returns:
    - pd.Series of cluster labels indexed by fan_id.
    """

    METHODS = {
        'kmeans':   learn_segments_kmeans,
        'nmf':      learn_segments_nmf,
        'item2vec': learn_segments_item2vec,
        'som':      learn_segments_som,
        'dt':       learn_segments_dt,
    }

    if method not in METHODS:
        raise ValueError(f"method must be one of {list(METHODS.keys())}")

    df = ratings_df.copy()

    if min_liked > 0:
        liked_counts = df[df['rating'] == 1].groupby('fan_id').size()
        valid_fans   = liked_counts[liked_counts >= min_liked].index
        df = df[df['fan_id'].isin(valid_fans)]

    return METHODS[method](df)

In [194]:
from sklearn.metrics import adjusted_rand_score

def run_all_methods(ratings_df: pd.DataFrame,
                    users_df: pd.DataFrame,
                    min_liked: int = 5) -> dict:
    """Run all 5 learn_segments methods and return ARI scores vs ground truth."""

    input_df = ratings_df.drop(columns=['fan_segment', 'reason'])

    gt = (ratings_df[['fan_id', 'fan_segment']]
          .drop_duplicates('fan_id')
          .set_index('fan_id')['fan_segment'])

    aris = {}
    for method in ['kmeans', 'nmf', 'item2vec', 'som', 'dt']:
        labels = learn_segments(input_df, method=method, min_liked=min_liked)
        common = labels.index.intersection(gt.index)
        aris[method] = round(adjusted_rand_score(gt.loc[common], labels.loc[common]), 3)
        print(f"  ARI: {aris[method]:.3f}\n")

    return aris

#### Experiments

We run all five methods under two conditions:

- **Run 1 - 5 attributes:** sparse signal, four segments partially or fully undetectable (Veteran Appreciators has no core attributes available; Defense Purists, Stat Nerds, and Playmaker Devotees each have only one of their two core attributes). Tests robustness under data scarcity.
- **Run 2 - 15 attributes:** full signal plus a red herring (`nationality`), which no segment uses. Tests whether methods pick up uninformative features.

Delta ARI measures how much each method benefits from the additional attributes.

In [195]:
np.random.seed(42); random.seed(42)
print("Run 1 — 5 attributes (sparse signal)")
players_5 = generate_entities(n=100, num_attributes=5)
users_5   = generate_users()
ratings_5 = generate_ratings(players_5, users_5, n_ratings_per_user=100)
aris_5    = run_all_methods(ratings_5, users_5)

Run 1 — 5 attributes (sparse signal)
5800 fans  (2000 / 1500 / 1000 / 800 / 500)
580000 ratings  |  seg1: 10.0%  seg2: 25.7%  seg3: 14.9%  seg4: 9.9%  seg5: 20.3%
[k-Means]
  group 1  (1038 fans, 11.7%)
    no clear preference found
  group 2  (962 fans, 12.4%)
    no clear preference found
  group 3  (1618 fans, 25.7%)
    no clear preference found
  group 4  (1192 fans, 12.7%)
    no clear preference found
  group 5  (922 fans, 11.2%)
    no clear preference found
  ARI: 0.287

[NMF]
  component 1  (148 fans, 14.0%)
    points_per_game > 21.58 AND points_per_game <= 23.91 AND field_goal_pct > 0.55
  component 2  (2239 fans, 13.5%)
    no clear preference found
  component 3  (938 fans, 16.9%)
    all_star_appearances > 3.50 AND assists_per_game > 8.10 AND points_per_game <= 15.43
  component 4  (1029 fans, 17.4%)
    all_star_appearances > 3.50 AND assists_per_game <= 1.80 AND points_per_game <= 24.93
  component 5  (1378 fans, 18.1%)
    all_star_appearances > 3.50 AND points_per_ga

#### Observations - Run 1

Highlight Reel is the only cleanly recoverable segment at 5 attributes since both its core attributes (points_per_game, all_star_appearances) fall within the first 5. Defense Purists, Stat Nerds, and Playmaker Devotees each have only one of their two core attributes available, producing a weak but detectable signal. Veteran Appreciators are completely invisible: neither age nor free_throw_pct appears at this level. k-Means and DT anchor on the available signal (~0.30). Item2Vec collapses to near zero since too few liked players per fan exist to learn meaningful co-occurrence patterns.

In [196]:
np.random.seed(42); random.seed(42)
print("Run 2 — 15 attributes (full signal + red herring)")
players_15 = generate_entities(n=100, num_attributes=15)
users_15   = generate_users()
ratings_15 = generate_ratings(players_15, users_15, n_ratings_per_user=100)
aris_15    = run_all_methods(ratings_15, users_15)

Run 2 — 15 attributes (full signal + red herring)
5800 fans  (2000 / 1500 / 1000 / 800 / 500)
580000 ratings  |  seg1: 30.5%  seg2: 27.4%  seg3: 20.5%  seg4: 18.2%  seg5: 23.7%
[k-Means]
  group 1  (1680 fans, 26.8%)
    all_star_appearances <= 3.50 AND points_per_game > 20.45 AND field_goal_pct <= 0.45
    all_star_appearances <= 3.50 AND points_per_game > 20.45 AND field_goal_pct > 0.45
    all_star_appearances > 3.50 AND three_point_pct <= 0.40 AND age <= 29.00
    all_star_appearances > 3.50 AND three_point_pct > 0.40 AND rebounds_per_game <= 2.48
    all_star_appearances > 3.50 AND three_point_pct > 0.40 AND rebounds_per_game > 2.48
  group 2  (2251 fans, 29.9%)
    field_goal_pct > 0.50 AND salary_tier > 0.50 AND rebounds_per_game <= 4.44
    field_goal_pct > 0.50 AND salary_tier > 0.50 AND rebounds_per_game > 4.44
  group 3  (796 fans, 18.2%)
    age > 30.50 AND free_throw_pct > 0.78 AND salary_tier <= 0.50
    age > 30.50 AND free_throw_pct > 0.78 AND salary_tier > 0.50
  group

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


[Item2Vec]
  cluster 1  (742 fans, 24.8%)
    assists_per_game > 6.43 AND rebounds_per_game > 8.11
  cluster 2  (2005 fans, 30.4%)
    field_goal_pct > 0.50 AND salary_tier > 0.50 AND rebounds_per_game <= 5.51
    field_goal_pct > 0.50 AND salary_tier > 0.50 AND rebounds_per_game > 5.51
  cluster 3  (1361 fans, 27.0%)
    all_star_appearances <= 3.50 AND points_per_game > 20.45 AND field_goal_pct <= 0.45
    all_star_appearances <= 3.50 AND points_per_game > 20.45 AND field_goal_pct > 0.45
    all_star_appearances > 3.50 AND three_point_pct <= 0.40 AND age <= 27.50
    all_star_appearances > 3.50 AND three_point_pct > 0.40 AND salary_tier <= 0.50
    all_star_appearances > 3.50 AND three_point_pct > 0.40 AND salary_tier > 0.50
  cluster 4  (907 fans, 20.5%)
    steals_per_game > 1.50 AND blocks_per_game > 1.05 AND three_point_pct <= 0.38
    steals_per_game > 1.50 AND blocks_per_game > 1.05 AND three_point_pct > 0.38
  cluster 5  (785 fans, 18.2%)
    age > 30.50 AND free_throw_pct > 0

#### Observations - Run 2

With full attributes all methods improve significantly. The most striking result is Item2Vec reaching ~0.86 — the highest of any method — despite never accessing player attributes directly. The co-like structure in the rating matrix encodes the same segment information as the explicit rules when interaction data is dense enough. k-Means and DT reach ~0.79. NMF reaches ~0.47; SOM ~0.35. The red herring (`nationality`) registers near-zero feature importance across all methods.

In [197]:
comparison = pd.DataFrame({
    'Method':         list(aris_5.keys()),
    'ARI (5 attrs)':  list(aris_5.values()),
    'ARI (15 attrs)': list(aris_15.values()),
})
comparison['Δ ARI'] = (comparison['ARI (15 attrs)'] - comparison['ARI (5 attrs)']).round(3)
comparison = comparison.sort_values('ARI (15 attrs)', ascending=False).reset_index(drop=True)
display(comparison)


,Method,ARI (5 attrs),ARI (15 attrs),Δ ARI
0,item2vec,0.001,0.857,0.856
1,kmeans,0.287,0.785,0.498
2,dt,0.287,0.785,0.498
3,nmf,0.058,0.470,0.412
4,som,0.149,0.345,0.196


### Summary

Attribute-based methods (k-Means, DT) are robust under data scarcity and perform well across both conditions. The surprise is Item2Vec: it collapses near zero at 5 attributes (too few co-likes to learn meaningful patterns) but surpasses every other method at 15 attributes — the largest Δ ARI of any approach. This is the collaborative filtering cold-start trade-off in action: once interaction data is dense, co-like patterns capture segment structure as effectively as explicit rules. NMF sits in the middle; SOM prioritizes geometric interpretability over raw accuracy.

In production recommender systems, these trade-offs motivate hybrid models that blend content-based and collaborative signals depending on data density.